# Hanno tenuto il mandato dichiarato?

Appena eletto, ogni Papa pronuncia un discorso **programmatico** — l'**omelia di
inizio pontificato** — in cui dichiara la sua linea. È detto *da Papa*, quindi
confrontarlo col resto del pontificato è legittimo (il limite del corpus: non
abbiamo i testi da cardinale, quindi non c'è un prima/dopo l'elezione — solo la
coerenza *dentro* il pontificato).

L'idea: prendere il **baricentro semantico** dell'omelia inaugurale come
*vettore-mandato*, e misurare anno per anno quanto i documenti del pontificato gli
restano vicini. La linea sale, resta, o si allontana?

## Come gira

Trovo l'omelia inaugurale di ogni Papa (per titolo), ne faccio il baricentro dei
chunk, e calcolo la similarità media per anno. Prodotto in **torch**.

In [ ]:
import numpy as np
import torch
from ponte import tabella

tab = tabella()
PAPI = ["Papa Giovanni Paolo II", "Papa Benedetto XVI", "Papa Francesco", "Papa Leone XIV"]
AB = ["GP2", "BXVI", "FRA", "LEO"]

t = tab.search().where("escludibile=false").select(["url", "papa", "data", "vector"]).limit(10**9).to_arrow()
papi = np.array(t.column("papa").to_pylist())
date = np.array([d[:4] if d else "" for d in t.column("data").to_pylist()])
V = torch.from_numpy(np.stack(t.column("vector").to_pylist()).astype("float32"))

# l'omelia di inizio pontificato, una per Papa
F = ("tipologia='homilies' AND lower(titolo) LIKE '%inizio%' AND "
     "(lower(titolo) LIKE '%pontificato%' OR lower(titolo) LIKE '%ministero%')")
man = tab.search().where(F).select(["url", "papa", "data", "titolo", "vector"]).limit(500).to_list()
mvec = {}
for p in PAPI:
    rs = [r for r in man if r["papa"] == p]
    if not rs:
        continue
    u0 = rs[0]["url"]; rs = [r for r in rs if r["url"] == u0]
    m = np.stack([r["vector"] for r in rs]).astype("float32").mean(0)
    mvec[p] = m / (np.linalg.norm(m) + 1e-9)
    print(f"{AB[PAPI.index(p)]:4} {rs[0]['data']}  ({len(rs)} chunk)  {rs[0]['titolo'][:46]}")

In [ ]:
def slope(xs, ys):           # pendenza lineare a mano (niente BLAS dopo torch)
    x = np.asarray(xs, "float64") - xs[0]; y = np.asarray(ys, "float64")
    xb, yb = x.mean(), y.mean()
    return float(((x - xb) * (y - yb)).sum() / (((x - xb) ** 2).sum() or 1))

print("tenuta del mandato — similarità media dei chunk dell'anno al vettore-mandato\n")
for p in PAPI:
    if p not in mvec:
        continue
    sims = (V @ torch.from_numpy(mvec[p].astype("float32"))).numpy()
    sel = papi == p
    rows = [(int(y), float(sims[sel & (date == y)].mean()))
            for y in sorted(set(date[sel])) if y and (sel & (date == y)).sum() >= 20]
    if not rows:
        continue
    ys = [s for _, s in rows]
    print(f"{AB[PAPI.index(p)]:4}: {rows[0][0]}={rows[0][1]:.3f} -> {rows[-1][0]}={rows[-1][1]:.3f}"
          f"   trend/anno {slope([y for y,_ in rows], ys):+.4f}   media {np.mean(ys):.3f}")

## Il risultato

| Papa | inizio | fine | trend/anno | media |
|---|--:|--:|--:|--:|
| GP2 | 0,912 (1978) | 0,902 (2005) | −0,0004 | 0,907 |
| BXVI | 0,907 (2005) | 0,912 (2013) | +0,0006 | 0,908 |
| FRA | 0,897 (2013) | 0,895 (2025) | −0,0001 | 0,895 |
| LEO | 0,907 (2025) | 0,906 (2026) | −0,0007 | 0,907 |

**Tutti tengono il mandato.** La fedeltà all'omelia inaugurale resta alta e
sostanzialmente **piatta** per l'intero pontificato — Giovanni Paolo II su
**27 anni** non si allontana (trend ≈ 0). Nessuna deriva: la linea dichiarata
all'inizio è quella di sempre.

## Come si legge — e i limiti (onesti)

- **Leggi il trend, non il livello.** Il valore assoluto ~0,90 è *gonfiato*: le
  similarità di e5 stanno tutte in alto (ogni testo religioso somiglia all'omelia).
  Il segnale affidabile è la **piattezza** — assenza di deriva — non il numero in sé.
- **Il "cosa" del mandato resta da scavare.** A livello di *famiglia* (notebook 04)
  le omelie inaugurali sono ~100% «fede e devozione»: troppo grosse per dire *quale*
  linea. Il passo dopo è isolare la **direzione distintiva** del mandato (ciò che
  l'omelia accentua oltre la media del Papa) e seguire *quella* nel tempo — così si
  vede crescere o calare il tema-firma, non solo "quanto somiglia".
- **Limite del corpus.** Vediamo solo documenti *da Papa*: misuriamo la coerenza
  *dentro* il pontificato, non un confronto con il prima (da cardinale), che non
  abbiamo.

---
*Solo misure **aggregate** (similarità medie per anno), nessun testo ripubblicato
(© Libreria Editrice Vaticana, fonte `url` nei metadati).*